In [1]:
!pip install transformers


In [3]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments, T5ForConditionalGeneration

In [6]:
train_data=pd.read_csv("/content/samsum-train.csv")


In [7]:
val_data=pd.read_csv("/content/samsum-validation.csv")

In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [39]:
train_data.iloc[0]["dialogue"]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [11]:
train_data.sample(10)

,id,dialogue,summary
5557,13729488,Jared: I really need to buy a new phone.\r\nSk...,Jared needs a new phone. Skyler suggests Jared...
8971,13863030,"Noah: I'm going to Ulla tonight, would you fan...",Mohammad and Noah are meeting at 5.30 at their...
2701,13717277,Christy: How are the test results?\r\nFred: We...,Scott has just got the test results and everyt...
7229,13730963,Hal: Xmas is coming\r\nDaphne: and ...\r\nHal:...,Hal wants to know which present Daphne would l...
11561,13716687,"Agnes: <file_video>\r\nMia: Haha, you're the w...",Mia should join Agnes and Fran when they go da...
6888,13864829,Fay: Hello! Collecting orders for the new cale...,Fay is taking orders for the new calendars. Ol...
7777,13730890,Fran: I can't find my boarding pass. Have you ...,Jim helps Fran find her boarding pass in the d...
11406,13811932,Ron: Omg I've just eaten the biggest bag of gu...,Ron has eaten a huge bag of gummy bears.
12295,13862228,Constanza: I was wondering.....\nConstanza: Ho...,Constanza's washing machine stopped working. S...
982,13682025,"Johana: Hi Yoli, this is my new number\r\nYoli...","Johana, Yoli and Ben met for dinner on Friday ..."


In [11]:
val_data.shape

(818, 3)

In [9]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

DATA PREPROCESSING

In [30]:
import re

def clean_data(text):
    # Replace carriage returns and newlines with a space
    text = re.sub(r"[\r\n]+", " ", text)
    # Remove speaker colon labels if needed, but for T5 dialogues, keeping names is usually fine.
    # Just normalize multiple spaces to a single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [31]:
train_data['dialogue']=train_data["dialogue"].apply(clean_data)
train_data['summary']=train_data["summary"].apply(clean_data)

val_data['dialogue']=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [32]:
train_data['dialogue'][0]

"Violet:hi!icameacrossthisAustin'sarticleandithoughtthatyoumightfinditinterestingViolet:<file_other>Claire:Hi!:)Thanks,butI'vealreadyreadit.:)Claire:Butthanksforthinkingaboutme:)"

TOKENIZER

In [12]:
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def tokenize_data(data):
    inputs = t5_tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True, return_attention_mask=True)
    target = t5_tokenizer(data["summary"], padding="max_length", max_length=128, truncation=True, return_attention_mask=True)

    inputs["labels"] = target["input_ids"]
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize_data, axis=1).tolist()
val_dataset = val_data.apply(tokenize_data, axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [28866, 10, 107, 23, 55, 2617, 526, 9, 11465, 8048, 14934, 17, 77, 31, 7, 8372, 232, 23, 11841, 17, 6279, 4188, 51, 2632, 8954, 155, 19405, 53, 553, 23, 32, 1655, 10, 2, 11966, 834, 9269, 3155, 254, 40, 2378, 10, 12146, 55, 10, 61, 17745, 7, 6, 2780, 196, 31, 162, 138, 23015, 5236, 155, 5, 10, 61, 254, 40, 2378, 10, 11836, 20349, 7, 1161, 10337, 53, 7932, 526, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [50]:
#input ids-dialogues-token ids
#attentions-mask->valid values and the padding value if token id=1 valid and id=0 the padding
#labels-target=>summery

In [16]:
len(train_dataset[0]["input_ids"])

512

Working with model

In [17]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
#fine-tuning

import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
import torch
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available, using CPU. Training will be very slow.")

GPU is available: Tesla T4


In [20]:
from transformers import TrainingArguments

#traning arguments
train_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_steps=1e6,
    warmup_steps=1e4,
    learning_rate=1e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
)

In [21]:
from transformers import Trainer, T5ForConditionalGeneration

# Ensure model is initialized
model = T5ForConditionalGeneration.from_pretrained("t5-small")

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [22]:
#train_model

trainer.train()

Epoch,Training Loss,Validation Loss
1,2.724010,1.746906
2,1.120672,0.952528
3,0.925159,0.799362
4,0.813543,0.748977
5,0.776222,0.717792
6,0.789616,0.695452


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3000, training_loss=2.3908192842801412, metrics={'train_runtime': 1406.5145, 'train_samples_per_second': 17.063, 'train_steps_per_second': 2.133, 'total_flos': 3248203235328000.0, 'train_loss': 2.3908192842801412, 'epoch': 6.0})

In [24]:
model.save_pretrained("./saved_model")
t5_tokenizer.save_pretrained("./saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_model/tokenizer_config.json', './saved_model/tokenizer.json')

In [25]:
model.from_pretrained("./saved_model")
t5_tokenizer.from_pretrained("./saved_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='./saved_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False, specia

In [ ]:
#testing the code logicb

In [33]:
def summerize_text(dialogue):
    dialogue = clean_data(dialogue)
    # The prefix 'summarize: ' is standard for T5
    inputs = t5_tokenizer.encode("summarize: " + dialogue, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate summary
    summary_ids = model.to(device).generate(
        inputs,
        max_length=150,
        num_beams=4,
        repetition_penalty=2.5,
        length_penalty=1.0,
        early_stopping=True
    )

    summary = t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [28]:
test_dialogue="""
Beatrice: I am in town, shopping. They have nice scarfs in the shop next to the church. Do you want one? Leo: No, thanks Beatrice: But you don't have a scarf.
Leo: Because I don't need it.
Beatrice: Last winter you had a cold all the time. A scarf could help.
 Leo: I don't like them.
  Beatrice: Actually, I don't care. You will get a scarf.
  Leo: How understanding of you!
  Beatrice: You were complaining the whole winter that you're going to die. I've had enough.
  Leo: Eh.
"""

summary=summerize_text(test_dialogue)
print(summary)

BeatriceandLeohavenicescarfsintheshopnexttothechurch.Theyhavenicescarfsintheshopnexttothechurch.


In [35]:
# Test Case 2: Business/Meeting coordination
test_dialogue_2 = """
James: Are we still on for the meeting at 3?
Sarah: I have a conflict. Can we push it to 4?
James: 4 PM works for me. Should I invite Mark?
Sarah: Yes, please. He needs to see the latest slides.
"""

# Test Case 3: Casual plans
test_dialogue_3 = """
Mike: Hey, do you want to grab pizza tonight?
Anna: I'd love to, but I'm going to the gym.
Mike: No worries, maybe tomorrow?
Anna: Tomorrow is perfect. Let's aim for 7 PM at Joe's.
"""

print("Summary 2:", summerize_text(test_dialogue_2))
print("Summary 3:", summerize_text(test_dialogue_3))

Summary 2: Sarah and Mark are still on for the meeting at 3pm.
Summary 3: Mike wants to grab pizza tonight, but she's going to the gym.


# Dialogue Summarization with T5-Small

This project focuses on fine-tuning the **T5-Small** (Text-to-Text Transfer Transformer) model for the task of dialogue summarization using the **SAMSum dataset**.

## Project Overview
The goal is to take conversational text (dialogues) and generate concise, human-like summaries. T5 is particularly well-suited for this as it treats every NLP task as a text-to-text problem.

## Dataset
- **Name:** SAMSum Corpus
- **Source:** [Hugging Face / SAMSum](https://huggingface.co/datasets/samsum)
- **Content:** Approximately 16k messenger-like dialogues with accompanying summaries.

## Implementation Steps
1.  **Preprocessing:**
    - Data cleaning using Regex to handle messenger-specific artifacts (newlines, carriage returns).
    - Normalizing whitespace to improve tokenizer efficiency.
2.  **Tokenization:**
    - Using `T5Tokenizer` with a max input length of 512 and max target length of 128.
    - Adding the `summarize: ` prefix to all input sequences as required by T5.
3.  **Training:**
    - **Framework:** Hugging Face `Trainer` API.
    - **Hardware:** Trained using NVIDIA Tesla T4 GPU (Google Colab).
    - **Hyperparameters:**
        - Epochs: 6
        - Learning Rate: 1e-4
        - Batch Size: 8
        - Weight Decay: 0.01
4.  **Evaluation:** Validation loss monitored per epoch.

## How to Use
To generate a summary for a new dialogue:

```python
# Load the saved model
model = T5ForConditionalGeneration.from_pretrained("./saved_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_model")

# Run the inference function
dialogue = "James: Are we still on for the meeting? Sarah: Yes, at 4 PM."
print(summerize_text(dialogue))
```